# Spoken numbers (Russian ASR) — interactive EDA

**Splits:** `train` (fit), `dev` (validation only — do not merge into train), `test` (inference / leaderboard only).

**Target:** `transcription` is an integer written as digits (e.g. `280520`); **digit count** = string length of that integer.

Set `DATA_ROOT` (or env `ASR2026_DATA`) to the folder that contains `train.csv` and the `train/`, `dev/`, `test/` audio subfolders.

In [1]:
from __future__ import annotations

import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import Audio, display
import ipywidgets as widgets
from scipy import stats
from tqdm.auto import tqdm

try:
    import librosa
except ImportError:
    librosa = None
import soundfile as sf

warnings.filterwarnings("ignore", category=UserWarning)

# Root folder with train.csv, dev.csv, test.csv and train/, dev/, test/ audio dirs
DATA_ROOT = Path("data/competitions/asr-2026-spoken-numbers-recognition-challenge").resolve()
print("DATA_ROOT =", DATA_ROOT)

DATA_ROOT = E:\ai_talent_hub\spoken-numbers-recognition\data\competitions\asr-2026-spoken-numbers-recognition-challenge


### Optional: download competition data (Kaggle Hub)

Requires Kaggle API credentials configured for your account. After download, set `DATA_ROOT` to the returned path (or set env `ASR2026_DATA`).

In [2]:
def assert_data_ready(root: Path) -> None:
    for name in ("train.csv", "dev.csv", "test.csv"):
        p = root / name
        if not p.is_file():
            raise FileNotFoundError(f"Missing {p}. Set DATA_ROOT or ASR2026_DATA.")

assert_data_ready(DATA_ROOT)


def audio_duration_sec(path: Path) -> float:
    path = Path(path)
    if librosa is not None:
        try:
            return float(librosa.get_duration(path=str(path)))
        except Exception:
            pass
    info = sf.info(str(path))
    return float(info.duration)


def load_split(name: str) -> pd.DataFrame:
    df = pd.read_csv(DATA_ROOT / f"{name}.csv")
    df["_split"] = name
    df["filepath"] = df["filename"].map(lambda x: DATA_ROOT / str(x))
    missing = ~df["filepath"].map(Path.is_file)
    if missing.any():
        bad = df.loc[missing, "filename"].head(5).tolist()
        raise FileNotFoundError(f"{missing.sum()} files missing under {DATA_ROOT}. Example: {bad}")
    durations = []
    for fp in tqdm(df["filepath"], desc=f"{name} duration"):
        durations.append(audio_duration_sec(fp))
    df["duration_sec"] = durations
    if "transcription" in df.columns:
        tr = df["transcription"]
        s = tr.astype("string")
        df["digit_len"] = s.str.len()
        df["digit_len"] = df["digit_len"].where(tr.notna(), np.nan)
    else:
        # Kaggle test CSV often omits the target column
        df["digit_len"] = np.nan
    dur = df["duration_sec"].astype(float)
    df["chars_per_sec"] = np.where(dur > 0, df["digit_len"].astype(float) / dur, np.nan)
    return df


train_df = load_split("train")
dev_df = load_split("dev")
test_df = load_split("test")

all_df = pd.concat([train_df, dev_df, test_df], ignore_index=True)
all_df.head()

train duration:   0%|          | 0/12553 [00:00<?, ?it/s]

dev duration:   0%|          | 0/2265 [00:00<?, ?it/s]

test duration:   0%|          | 0/2582 [00:00<?, ?it/s]

,filename,transcription,spk_id,gender,ext,samplerate,_split,filepath,duration_sec,digit_len,chars_per_sec
0,train/0007c21c23.wav,139473.0,spk_E,female,wav,24000,train,E:\ai_talent_hub\spoken-numbers-recognition\da...,3.044125,6.0,1.971010
1,train/000bee1b1d.wav,992597.0,spk_B,male,wav,24000,train,E:\ai_talent_hub\spoken-numbers-recognition\da...,3.217500,6.0,1.864802
2,train/001a718902.wav,500869.0,spk_A,female,wav,22050,train,E:\ai_talent_hub\spoken-numbers-recognition\da...,3.513333,6.0,1.707780
3,train/001e8e5565.wav,969908.0,spk_C,male,wav,22050,train,E:\ai_talent_hub\spoken-numbers-recognition\da...,3.339184,6.0,1.796846
4,train/001ee5be6b.wav,80484.0,spk_E,female,wav,24000,train,E:\ai_talent_hub\spoken-numbers-recognition\da...,3.102208,5.0,1.611755


In [3]:
# Quick numeric summary (labeled splits only for target stats)
for name, g in [("train", train_df), ("dev", dev_df), ("test", test_df)]:
    print(f"=== {name} === rows={len(g)}")
    print(g[["duration_sec", "digit_len", "chars_per_sec"]].describe())

=== train === rows=12553
       duration_sec  digit_len  chars_per_sec
count  12553.000000    12553.0   12553.000000
mean       3.071296   5.888553       1.964605
std        0.512530   0.353112       0.316556
min        1.070375        2.0       1.049092
25%        2.753333        6.0       1.785338
50%        3.037417        6.0       1.945341
75%        3.316500        6.0       2.127597
max        5.719229        6.0       4.411765
=== dev === rows=2265
       duration_sec  digit_len  chars_per_sec
count   2265.000000     2265.0    2265.000000
mean       3.374891   5.891832       1.832866
std        2.169037   0.341801       0.355294
min        1.400000        4.0       0.059368
25%        2.880000        6.0       1.587144
50%        3.279875        6.0       1.796844
75%        3.720000        6.0       2.032520
max      101.064000        6.0       3.338666
=== test === rows=2582
       duration_sec  digit_len  chars_per_sec
count   2582.000000        0.0            0.0
mean      

In [4]:
def signed_z_scores(x: pd.Series) -> pd.Series:
    """Per-value z-score (signed); NaN where undefined."""
    x = x.astype(float)
    valid = x.notna()
    if not valid.any():
        return pd.Series(np.nan, index=x.index)
    mu, sd = float(x[valid].mean()), float(x[valid].std(ddof=0))
    if sd == 0:
        return pd.Series(np.nan, index=x.index)
    return (x - mu) / sd


def residual_signed_z(duration: pd.Series, digit_len: pd.Series) -> pd.Series:
    """Signed z of OLS residual (duration ~ digit_len). NaN where undefined."""
    mask = digit_len.notna() & duration.notna()
    if not mask.any():
        return pd.Series(np.nan, index=duration.index)
    d = duration[mask].astype(float)
    L = digit_len[mask].astype(float)
    slope, intercept, _, _, _ = stats.linregress(L.values, d.values)
    pred = intercept + slope * L
    resid = d - pred
    mu, sd = float(resid.mean()), float(resid.std(ddof=0))
    if sd == 0:
        return pd.Series(np.nan, index=duration.index)
    z = (resid - mu) / sd
    full = pd.Series(np.nan, index=duration.index)
    full.loc[mask] = z.values
    return full


def attach_outlier_flags(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    cps_z = signed_z_scores(out["chars_per_sec"])
    rz = residual_signed_z(out["duration_sec"], out["digit_len"])
    for sig in (3, 4, 5):
        v = cps_z.notna()
        out[f"outlier_cps_{sig}sigma"] = (cps_z.abs() > sig) & v
        out[f"outlier_cps_left_{sig}sigma"] = (cps_z < -sig) & v
        out[f"outlier_cps_right_{sig}sigma"] = (cps_z > sig) & v
        rv = rz.notna()
        out[f"outlier_resid_{sig}sigma"] = (rz.abs() > sig) & rv
        out[f"outlier_resid_left_{sig}sigma"] = (rz < -sig) & rv
        out[f"outlier_resid_right_{sig}sigma"] = (rz > sig) & rv
    return out


all_flagged = attach_outlier_flags(all_df)
train_flagged = attach_outlier_flags(train_df)
dev_flagged = attach_outlier_flags(dev_df)

In [5]:
# Duration histograms by split (interactive)
fig = go.Figure()
for split, sub in [("train", train_df), ("dev", dev_df), ("test", test_df)]:
    fig.add_trace(
        go.Histogram(
            x=sub["duration_sec"],
            name=split,
            opacity=0.65,
            nbinsx=40,
        )
    )
fig.update_layout(
    barmode="overlay",
    title="Audio duration (s) by split",
    xaxis_title="duration (s)",
    yaxis_title="count",
    legend_title_text="split",
)
fig.show()

In [ ]:
def plot_digit_vs_duration(df: pd.DataFrame, title: str, sigma: int = 3):
    sub = df[df["digit_len"].notna()].copy()
    cps_flag = sub[f"outlier_cps_{sigma}sigma"]
    resid_flag = sub[f"outlier_resid_{sigma}sigma"]
    any_flag = cps_flag | resid_flag

    fig = make_subplots(
        rows=1,
        cols=2,
        subplot_titles=(
            f"Digit length vs duration ({sigma}σ outliers highlighted)",
            f"Chars/sec distribution ({sigma}σ on rate)",
        ),
    )

    fig.add_trace(
        go.Scatter(
            x=sub.loc[~any_flag, "digit_len"],
            y=sub.loc[~any_flag, "duration_sec"],
            mode="markers",
            marker=dict(size=5, opacity=0.35),
            name="inlier",
            hovertext=sub.loc[~any_flag, "filename"],
        ),
        row=1,
        col=1,
    )
    fig.add_trace(
        go.Scatter(
            x=sub.loc[any_flag, "digit_len"],
            y=sub.loc[any_flag, "duration_sec"],
            mode="markers",
            marker=dict(size=7, color="red", opacity=0.85),
            name=f"outlier (≥{sigma}σ)",
            hovertext=sub.loc[any_flag, "filename"],
        ),
        row=1,
        col=1,
    )

    mu = float(sub["chars_per_sec"].mean())
    sd = float(sub["chars_per_sec"].std(ddof=0))
    fig.add_trace(
        go.Histogram(x=sub["chars_per_sec"], nbinsx=50, name="chars/s", opacity=0.7),
        row=1,
        col=2,
    )
    for sign in (-1, 1):
        fig.add_vline(
            x=mu + sign * sigma * sd,
            line_dash="dash",
            line_color="red",
            annotation_text=f"{sign:+} {sigma}σ",
            row=1,
            col=2,
        )
    fig.add_vline(x=mu, line_color="green", annotation_text="mean", row=1, col=2)

    fig.update_xaxes(title_text="digit count (string length)", row=1, col=1)
    fig.update_yaxes(title_text="duration (s)", row=1, col=1)
    fig.update_xaxes(title_text="digits / second", row=1, col=2)

    fig.update_layout(title_text=title, showlegend=True, height=480)
    fig.show()


_FLAGGED_BY_SPLIT = {"train": train_flagged, "dev": dev_flagged, "all": all_flagged}

sigma_radio = widgets.RadioButtons(options=[3, 4, 5], description="σ cutoff")
split_dd = widgets.Dropdown(options=["train", "dev", "all"], value="train", description="Split")


def _eda_refresh(change=None):
    plot_digit_vs_duration(
        _FLAGGED_BY_SPLIT[split_dd.value],
        title=f"EDA — {split_dd.value} (σ={sigma_radio.value})",
        sigma=sigma_radio.value,
    )


sigma_radio.observe(_eda_refresh, names="value")
split_dd.observe(_eda_refresh, names="value")

display(widgets.HBox([sigma_radio, split_dd]))
_eda_refresh()

In [13]:
# Listen to random outliers (chars/sec z-score or duration–length residual)


def pick_and_play(
    split_name: str,
    mode: str,
    sigma: int,
    tail: str,
    seed: int | None = None,
):
    mapping = {"train": train_flagged, "dev": dev_flagged, "all": all_flagged}
    df = mapping[split_name]
    prefix = "cps" if mode == "chars_per_sec" else "resid"
    if tail == "both":
        col = f"outlier_{prefix}_{sigma}sigma"
    else:
        col = f"outlier_{prefix}_{tail}_{sigma}sigma"
    sub = df[df[col] & df["filepath"].notna()]
    if sub.empty:
        print("No outliers for this setting.")
        return
    row = sub.sample(1, random_state=seed).iloc[0]
    fp = row["filepath"]
    print("split:", row["_split"], "| file:", row["filename"])
    print("transcription:", row.get("transcription"), "| digits:", row.get("digit_len"))
    print(f"duration_sec={row['duration_sec']:.3f} | chars/s={row['chars_per_sec']:.3f}")
    display(Audio(str(fp)))


mode_dd = widgets.Dropdown(
    options=[
        ("chars_per_sec z on rate", "chars_per_sec"),
        ("residual z (duration vs digits)", "resid"),
    ],
    description="Outlier type",
)
split_play = widgets.Dropdown(options=["train", "dev", "all"], value="train", description="Split")
sigma_play = widgets.RadioButtons(options=[3, 4, 5], description="σ")
tail_dd = widgets.Dropdown(
    options=[
        ("Both tails (|z| > σ)", "both"),
        ("Left tail (low z)", "left"),
        ("Right tail (high z)", "right"),
    ],
    value="both",
    description="Cutoff",
)
seed_txt = widgets.IntText(value=0, description="seed (0=random)")

btn = widgets.Button(description="Play random outlier", button_style="primary")


def on_click(_):
    seed = None if seed_txt.value == 0 else int(seed_txt.value)
    pick_and_play(
        split_play.value,
        mode_dd.value,
        sigma_play.value,
        tail_dd.value,
        seed=seed,
    )


btn.on_click(on_click)
display(
    widgets.VBox(
        [
            widgets.HBox([split_play, mode_dd, tail_dd, sigma_play, seed_txt]),
            btn,
            widgets.HTML(
                "<i>Chars/s: left = slower than typical, right = faster. "
                "Residual: left = shorter duration than line fit, right = longer. "
                "Seed 0 = new random draw each click.</i>"
            ),
        ]
    )
)

In [15]:
# Play train audio samples where digit length < 3

# Filter samples in the train split with digit length < 3
short_digits_df = all_df[(all_df["_split"] == "train") & (all_df["digit_len"] < 3)]

# Play up to 5 random samples (change n as desired)
n = min(5, len(short_digits_df))
for _, row in short_digits_df.sample(n).iterrows():
    print(f'Filename: {row["filename"]}, Digit length: {row["digit_len"]}, Transcription: {row["transcription"]}')
    display(Audio(filename=row["filepath"]))

Filename: train/c7d2263e89.wav, Digit length: 2.0, Transcription: 61.0


Filename: train/ee643401ea.wav, Digit length: 2.0, Transcription: 14.0


Filename: train/1339fd4329.wav, Digit length: 2.0, Transcription: 19.0


In [14]:
all_df.groupby("_split")["gender"].value_counts().sort_index()

_split  gender
dev     female     988
        male      1277
train   female    7665
        male      4888
Name: count, dtype: int64

In [10]:
all_df.groupby("_split")["samplerate"].value_counts().sort_index()

_split  samplerate
dev     16000          2265
test    16000          2582
train   22050          2129
        24000         10424
Name: count, dtype: int64

In [11]:
all_df.groupby("_split")["ext"].value_counts().sort_index()

_split  ext
dev     mp3      688
        wav     1577
test    mp3      812
        wav     1770
train   wav    12553
Name: count, dtype: int64